# PPE-Detection fine-tuning (best.pt uzerine, dusuk LR)

Bu notebook mevcut `best.pt` modelini SIFIRDAN egitmez; dusuk learning rate
kullanarak fine-tune eder. Orijinal `best.pt` dosyasina hicbir asamada yazma
yapilmaz; cikti her zaman ayri bir isimle (`hardhat_v2.pt`) kaydedilir.

**Roboflow kredi notu:** Workspace'in generate/version-download kredisi
tukendigi (0 kredi, Ekim 1'de reset) icin dataset Roboflow'un ucretli
`generate` pipeline'i UZERINDEN degil, kredi gerektirmeyen `search` +
`images/:id` (image detail) API'leri uzerinden dogrudan indiriliyor.
Split (70/20/10) zaten proje seviyesinde ayarlandi (rebalance API'siyle);
bu notebook o split atamasini oldugu gibi kullaniyor. Augmentation da
Roboflow yerine burada, yerelde (Albumentations ile) x2 olarak uygulaniyor
-> hicbir adim kredi harcamiyor.

In [ ]:
!pip install -q ultralytics albumentations opencv-python-headless tqdm

In [ ]:
# --- Ayarlar ---
from getpass import getpass
API_KEY = getpass("Roboflow API key: ")
WORKSPACE = "remzi-taskin"
PROJECT = "ppe-detection-o17cs"

# best.pt ile AYNI sira/ID (config.yaml: "best.pt: {0:'Helmet',1:'No-Helmet Head'}")
CLASS_ORDER = ["Helmet", "No-Helmet Head"]

DATA_DIR = "dataset_raw"   # indirilen ham (augmentsiz) veri buraya duser
AUG_VERSIONS_PER_IMAGE = 1  # train split'e kac EK augmentli kopya uretilsin -> toplam x(1+N) = x2

IMG_SIZE = 768          # SAHI slice boyutuyla tutarli (config.yaml: sahi.slice_height/width)
EPOCHS = 50             # fine-tune, sifirdan egitim degil -> dusuk epoch yeterli
PATIENCE = 15           # erken durdurma: N epoch boyunca iyilesme yoksa dur
LR0 = 0.001             # ultralytics default 0.01'in ~1/10'u
LRF = 0.01              # cosine decay ile inecegi son oran (efektif min lr = LR0*LRF)
FREEZE = 10             # backbone'un ilk N katmanini dondur (kucuk/dengesiz veri -> overfit/catastrophic forgetting riskini azaltir)
BATCH = 16

## 1) Orijinal best.pt'yi yukle (dokunulmaz kopya)

Sunucudaki `sunucu/detector/models/best.pt` dosyasini bu hucreyle Colab'a
yukle (sol panel > Files > Upload, ya da Drive baglayip oradan kopyala).
Bu dosya SADECE fine-tuning'in baslangic noktasi (pretrained weights) olarak
okunur, uzerine hicbir zaman yazilmaz.

In [ ]:
from google.colab import files
import shutil, os

os.makedirs("base_model", exist_ok=True)
uploaded = files.upload()  # best.pt sec
src_name = list(uploaded.keys())[0]
BASE_WEIGHTS = "base_model/base_best.pt"
shutil.move(src_name, BASE_WEIGHTS)
print("Baslangic agirligi (salt okunur referans):", BASE_WEIGHTS)

## 2) 1274 gorseli Roboflow'dan KREDI HARCAMADAN indir

`generate`/`version.download()` yerine `/search` (liste, split bilgisiyle)
ve `/images/:id` (mutlak piksel bbox + orijinal URL) API'leri kullanilir.
Bu ikisi Roboflow'un ucretli generate pipeline'inin DISINDA, kredi
tuketmeyen yonetim/inceleme uclaridir. Split atamasi (`train`/`valid`/`test`)
projeye onceden uygulanan 70/20/10 rebalance'tan aynen gelir.

In [ ]:
import requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm.auto import tqdm

BASE_URL = f"https://api.roboflow.com/{WORKSPACE}/{PROJECT}"


def make_session():
    s = requests.Session()
    retry = Retry(total=4, backoff_factor=1.5, status_forcelist=[429, 500, 502, 503, 504])
    adapter = HTTPAdapter(max_retries=retry, pool_maxsize=20)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    return s


SESSION = make_session()
REQ_TIMEOUT = (10, 25)  # (connect, read) saniye -- ayri tuple: takilma durumunda kesin patlar


def list_all_images():
    images, offset, limit = [], 0, 250
    while True:
        r = SESSION.post(
            f"{BASE_URL}/search",
            params={"api_key": API_KEY},
            json={"in_dataset": True, "limit": limit, "offset": offset, "fields": ["id", "split"]},
            timeout=REQ_TIMEOUT,
        )
        r.raise_for_status()
        data = r.json()
        images.extend(data["results"])
        offset += limit
        if offset >= data["total"]:
            break
    return images



def fetch_and_save(entry, out_dir: Path):
    image_id, split = entry["id"], entry["split"]
    marker = out_dir / ".fetched" / f"{image_id}.done"
    if marker.exists():
        return  # onceki calistirmadan zaten indirilmis -> atla (resumable)

    detail = SESSION.get(
        f"{BASE_URL}/images/{image_id}", params={"api_key": API_KEY}, timeout=REQ_TIMEOUT
    ).json()["image"]
    img_bytes = SESSION.get(detail["urls"]["original"], timeout=REQ_TIMEOUT).content

    ann = detail["annotation"]
    w, h = ann["width"], ann["height"]
    stem = Path(detail["name"]).stem

    img_path = out_dir / "images" / split / f"{stem}.jpg"
    img_path.parent.mkdir(parents=True, exist_ok=True)
    img_path.write_bytes(img_bytes)

    lines = []
    for box in ann["boxes"]:
        cls_id = CLASS_ORDER.index(box["label"])
        x, y = float(box["x"]), float(box["y"])
        bw, bh = float(box["width"]), float(box["height"])
        lines.append(f"{cls_id} {x / w:.6f} {y / h:.6f} {bw / w:.6f} {bh / h:.6f}")

    lbl_path = out_dir / "labels" / split / f"{stem}.txt"
    lbl_path.parent.mkdir(parents=True, exist_ok=True)
    lbl_path.write_text("\n".join(lines))

    marker.parent.mkdir(parents=True, exist_ok=True)
    marker.write_text("ok")


def run_download_pass(entries, out_dir, max_workers, per_item_timeout=60):
    failed = []
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        future_to_entry = {pool.submit(fetch_and_save, e, out_dir): e for e in entries}
        for fut in tqdm(as_completed(future_to_entry), total=len(future_to_entry), desc="indiriliyor"):
            entry = future_to_entry[fut]
            try:
                fut.result(timeout=per_item_timeout)  # gercek zaman aşımı garantisi (thread askida kalsa bile loop devam eder)
            except Exception as exc:
                print(f"  [ATLANDI] {entry['id']} ({entry['split']}): {exc!r}")
                failed.append(entry)
    return failed


out_dir = Path(DATA_DIR)
entries = list_all_images()
print(f"Toplam gorsel: {len(entries)} (beklenen: 1274)")

# Dusuk paralellik: Roboflow CDN bazi Colab IP'lerinde yogun paralel istekte
# takiliyor/rate-limit uyguluyor. 4 worker + kesin (connect,read) timeout +
# resumable "already done" isaretleyici ile calisiyoruz.
failed = run_download_pass(entries, out_dir, max_workers=4)

if failed:
    print(f"\n{len(failed)} gorsel ilk geciste basarisiz oldu, tek tek (max_workers=1) tekrar deneniyor...")
    still_failed = run_download_pass(failed, out_dir, max_workers=1, per_item_timeout=90)
    if still_failed:
        print(f"UYARI: {len(still_failed)} gorsel iki denemede de basarisiz: {[e['id'] for e in still_failed]}")

for split in ("train", "valid", "test"):
    n = len(list((out_dir / "images" / split).glob("*.jpg"))) if (out_dir / "images" / split).exists() else 0
    print(split, "->", n, "gorsel")

## 3) Yerel augmentation (x2 hedefi, sabit kamera acisina gore)

Sadece **train** split'e uygulanir (valid/test temiz kalir, gercek olcum
bozulmaz). Kamera sabit oldugu icin agresif geometrik augmentation (rotate/
shear/perspective) EKLENMEZ -> gercekte hic olmayacak varyasyon, sadece
gurultu katar. Fotometrik varyasyona (aydinlatma/pozlama/hafif bulaniklik)
agirlik verilir -> gun/gece, hava, kamera gurultusu gibi gercek sahne
degisimlerini simule eder. `AUG_VERSIONS_PER_IMAGE=1` -> train seti x2'ye
cikar (orijinal + 1 augmentli kopya).

In [ ]:
import albumentations as A
import cv2

transform = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.15, p=0.8),
        A.RandomGamma(gamma_limit=(80, 120), p=0.5),  # pozlama varyasyonu
        A.GaussianBlur(blur_limit=(3, 5), p=0.3),
        A.GaussNoise(std_range=(0.02, 0.08), p=0.3),
    ],
    bbox_params=A.BboxParams(format="yolo", label_fields=["class_ids"], min_visibility=0.2),
)


def load_yolo_labels(lbl_path: Path):
    if not lbl_path.exists() or not lbl_path.read_text().strip():
        return [], []
    boxes, class_ids = [], []
    for line in lbl_path.read_text().strip().splitlines():
        cls, x, y, w, h = line.split()
        boxes.append([float(x), float(y), float(w), float(h)])
        class_ids.append(int(cls))
    return boxes, class_ids


train_images = sorted((out_dir / "images" / "train").glob("*.jpg"))
for img_path in tqdm(train_images, desc="augment (train)"):
    lbl_path = out_dir / "labels" / "train" / f"{img_path.stem}.txt"
    boxes, class_ids = load_yolo_labels(lbl_path)
    if not boxes:
        continue  # bbox'siz kare augment edilmez (bos label uretmemek icin)

    img = cv2.imread(str(img_path))
    for v in range(AUG_VERSIONS_PER_IMAGE):
        result = transform(image=img, bboxes=boxes, class_ids=class_ids)
        if not result["bboxes"]:
            continue  # augmentasyon sonrasi tum bbox'lar disari tastiysa atla

        aug_stem = f"{img_path.stem}_aug{v}"
        cv2.imwrite(str(out_dir / "images" / "train" / f"{aug_stem}.jpg"), result["image"])
        lines = [
            f"{cid} {b[0]:.6f} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f}"
            for b, cid in zip(result["bboxes"], result["class_ids"])
        ]
        (out_dir / "labels" / "train" / f"{aug_stem}.txt").write_text("\n".join(lines))

n_train_final = len(list((out_dir / "images" / "train").glob("*.jpg")))
print(f"Train seti (augment sonrasi): {n_train_final} gorsel")

In [ ]:
data_yaml = f"""\
path: {out_dir.resolve()}
train: images/train
val: images/valid
test: images/test
names:
  0: {CLASS_ORDER[0]}
  1: {CLASS_ORDER[1]}
"""
DATA_YAML = str((out_dir / "data.yaml").resolve())
Path(DATA_YAML).write_text(data_yaml)
print(data_yaml)

## 4) Fine-tune (base_best.pt uzerinden, dusuk LR)

`model = YOLO(BASE_WEIGHTS)` -> sifirdan degil, mevcut agirliklardan devam
(warm start). Cikti `finetune_run/hardhat_v2/weights/best.pt` altina duser;
`base_best.pt` bu asamada degismez.

In [ ]:
from ultralytics import YOLO

model = YOLO(BASE_WEIGHTS)
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    lr0=LR0,
    lrf=LRF,
    freeze=FREEZE,
    patience=PATIENCE,
    project="finetune_run",
    name="hardhat_v2",
    exist_ok=True,
)
NEW_WEIGHTS = "finetune_run/hardhat_v2/weights/best.pt"
print("Yeni agirlik:", NEW_WEIGHTS)

## 5) Karsilastirma: eski vs yeni model, ayni test split uzerinde

Genel mAP degil, **No-Helmet Head** sinifinin precision/recall degerine bak.
Asil sorduğumuz soru bu: ihlal tespiti (recall) iyilesti mi?

In [ ]:
def per_class_report(weights_path, label):
    m = YOLO(weights_path)
    r = m.val(data=DATA_YAML, split="test", imgsz=IMG_SIZE)
    print(f"\n=== {label} ({weights_path}) ===")
    for i, name in r.names.items():
        print(f"  {name:20s}  P={r.box.p[i]:.3f}  R={r.box.r[i]:.3f}  mAP50={r.box.ap50[i]:.3f}")
    return r

old_r = per_class_report(BASE_WEIGHTS, "ESKI (best.pt)")
new_r = per_class_report(NEW_WEIGHTS, "YENI (hardhat_v2)")

## 6) Yeni agirligi indir

Metrikler tatmin ediciyse (ozellikle No-Helmet Head recall dususu yoksa/
artissa) `hardhat_v2.pt`'yi indir, sunucuda `detector/models/best.pt`'yi
MANUEL olarak degistir (bu notebook o dosyaya dokunmaz).

In [ ]:
from google.colab import files
shutil.copy(NEW_WEIGHTS, "hardhat_v2.pt")
files.download("hardhat_v2.pt")

## 7) (Opsiyonel) Ornek video/kare uzerinde gorsel dogrulama

Sahadan bir kac ornek kareyi Colab'a yukleyip iki modelin tahminlerini
yan yana koy. Sayisal metrik iyi cikip sahada tutarsizlik varsa (ör. gece
IR karelerinde False Positive artisi) bu asamada gorulur.

In [ ]:
import glob

sample_dir = "sample_frames"  # birkac test karesini bu klasore yukle
os.makedirs(sample_dir, exist_ok=True)
uploaded = files.upload()
for name in uploaded:
    shutil.move(name, os.path.join(sample_dir, name))

old_model = YOLO(BASE_WEIGHTS)
new_model = YOLO(NEW_WEIGHTS)
for img_path in glob.glob(f"{sample_dir}/*"):
    print("\n---", img_path, "---")
    old_model.predict(img_path, imgsz=IMG_SIZE, conf=0.3, save=True, project="preview", name="old", exist_ok=True)
    new_model.predict(img_path, imgsz=IMG_SIZE, conf=0.3, save=True, project="preview", name="new", exist_ok=True)
print("Karsilastirma goruntuleri: preview/old ve preview/new klasorlerinde")